<a href="https://colab.research.google.com/github/nuuuruur/nuuruuur/blob/main/Images_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Compressor to Target File Size
This tool allows you to upload an image and compress it until it reaches a specific size in KB or MB.

In [ ]:
import os
import io
from PIL import Image
from google.colab import files

def compress_image(image_path, target_size_kb):
    """Compresses an image to a target size in KB."""
    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')

    quality = 95
    output_io = io.BytesIO()

    # Iteratively find the right quality
    while quality > 5:
        output_io.seek(0)
        output_io.truncate(0)
        img.save(output_io, format='JPEG', quality=quality, optimize=True)
        current_size = output_io.tell() / 1024
        if current_size <= target_size_kb:
            break
        quality -= 5

    return output_io.getvalue()

# 1. Upload file
print("Please upload the image you want to compress:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]

    # 2. Get target size from user
    size_input = input("Enter target size (e.g., 500KB or 2MB): ").strip().upper()

    if 'MB' in size_input:
        target_kb = float(size_input.replace('MB', '')) * 1024
    else:
        target_kb = float(size_input.replace('KB', ''))

    # 3. Compress
    print(f"Compressing {filename} to approximately {size_input}...")
    compressed_data = compress_image(filename, target_kb)

    # 4. Save and Download
    output_filename = f"compressed_{filename.split('.')[0]}.jpg"
    with open(output_filename, 'wb') as f:
        f.write(compressed_data)

    print(f"Done! Compressed size: {len(compressed_data)/1024:.2f} KB")
    files.download(output_filename)
else:
    print("No file uploaded.")

Please upload the image you want to compress:


Saving Muhammad Hafizh Nur Dzaki.jpg to Muhammad Hafizh Nur Dzaki.jpg
Enter target size (e.g., 500KB or 2MB): 199KB
Compressing Muhammad Hafizh Nur Dzaki.jpg to approximately 199KB...
Done! Compressed size: 190.99 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Batch Image Compression
Use the cell below to upload multiple images and compress them all to a specific target size. Results will be downloaded as a ZIP archive.

In [ ]:
import zipfile

def batch_compress():
    print("Please upload the images you want to compress (you can select multiple):")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded.")
        return

    size_input = input("Enter target size for EACH image (e.g., 500KB or 1MB): ").strip().upper()
    if 'MB' in size_input:
        target_kb = float(size_input.replace('MB', '')) * 1024
    else:
        target_kb = float(size_input.replace('KB', ''))

    zip_filename = "compressed_images.zip"

    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for filename in uploaded.keys():
            print(f"Compressing {filename}...")
            compressed_data = compress_image(filename, target_kb)

            # Create a name for the file inside the zip
            inner_name = f"compressed_{os.path.splitext(filename)[0]}.jpg"
            zipf.writestr(inner_name, compressed_data)

    print(f"\nDone! All images compressed and added to {zip_filename}")
    files.download(zip_filename)

# Run the batch process
batch_compress()

### Resize Images by Width
This tool allows you to upload images and resize them to a specific width while maintaining their aspect ratio.

In [ ]:
import os
import zipfile
from PIL import Image
from google.colab import files
import io

def resize_by_width():
    print("Please upload the images you want to resize:")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded.")
        return

    try:
        target_width = int(input("Enter target width in pixels (e.g., 2000): ").strip().lower().replace('px', ''))
    except ValueError:
        print("Invalid input. Please enter a number.")
        return

    num_files = len(uploaded)

    if num_files == 1:
        # Handle single file: resize and download directly
        filename, data = list(uploaded.items())[0]
        print(f"Resizing {filename}...")
        img = Image.open(io.BytesIO(data))

        width_percent = (target_width / float(img.size[0]))
        target_height = int((float(img.size[1]) * float(width_percent)))

        resized_img = img.resize((target_width, target_height), Image.Resampling.LANCZOS)

        output_filename = f"resized_{target_width}px_{filename}"
        save_format = 'JPEG' if img.format in ['JPEG', 'JPG'] else img.format or 'JPEG'
        resized_img.convert('RGB').save(output_filename, format=save_format, quality=90)

        print(f"\nDone! Resized image saved as {output_filename}")
        files.download(output_filename)

    else:
        # Handle multiple files: zip them
        zip_filename = f"resized_{target_width}px_images.zip"
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for filename, data in uploaded.items():
                print(f"Resizing {filename}...")
                img = Image.open(io.BytesIO(data))

                width_percent = (target_width / float(img.size[0]))
                target_height = int((float(img.size[1]) * float(width_percent)))

                resized_img = img.resize((target_width, target_height), Image.Resampling.LANCZOS)

                img_byte_arr = io.BytesIO()
                save_format = 'JPEG' if img.format in ['JPEG', 'JPG'] else img.format or 'JPEG'
                resized_img.convert('RGB').save(img_byte_arr, format=save_format, quality=90)

                inner_name = f"resized_{target_width}px_{filename}"
                zipf.writestr(inner_name, img_byte_arr.getvalue())

        print(f"\nDone! All images resized and added to {zip_filename}")
        files.download(zip_filename)

# Run the resizing process
resize_by_width()

Please upload the images you want to resize:


Saving Muhammad Hafizh Nur Dzaki.jpg to Muhammad Hafizh Nur Dzaki.jpg


KeyboardInterrupt: Interrupted by user

### Resize Images to Specific Dimensions (Width x Height)
This tool allows you to resize images to an exact pixel size (e.g., 400x600).

In [ ]:
import os
import zipfile
from PIL import Image
from google.colab import files
import io

def resize_to_dimensions():
    print("Please upload the images you want to resize:")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded.")
        return

    dim_input = input("Enter maximum dimensions as Width x Height (e.g., 400x600): ").strip().lower()
    try:
        if 'x' in dim_input:
            target_width, target_height = map(int, dim_input.split('x'))
        else:
            print("Invalid format. Please use 'Width x Height' (e.g., 400x600).")
            return
    except ValueError:
        print("Invalid input. Please enter numbers for width and height.")
        return

    num_files = len(uploaded)

    if num_files == 1:
        filename, data = list(uploaded.items())[0]
        img = Image.open(io.BytesIO(data))

        # Use thumbnail to ensure dimensions are <= target while maintaining aspect ratio
        img.thumbnail((target_width, target_height), Image.Resampling.LANCZOS)
        new_w, new_h = img.size

        print(f"Resizing {filename} to {new_w}x{new_h} (max bounds: {target_width}x{target_height})...")

        output_filename = f"resized_{new_w}x{new_h}_{filename}"
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")

        img.save(output_filename, format='JPEG', quality=95)
        print(f"Done! Download starting...")
        files.download(output_filename)
    else:
        zip_filename = f"resized_batch_images.zip"
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for filename, data in uploaded.items():
                img = Image.open(io.BytesIO(data))
                img.thumbnail((target_width, target_height), Image.Resampling.LANCZOS)
                new_w, new_h = img.size

                print(f"Resizing {filename} to {new_w}x{new_h}...")

                img_byte_arr = io.BytesIO()
                if img.mode in ("RGBA", "P"):
                    img = img.convert("RGB")
                img.save(img_byte_arr, format='JPEG', quality=95)

                inner_name = f"resized_{new_w}x{new_h}_{filename}"
                zipf.writestr(inner_name, img_byte_arr.getvalue())

        print(f"\nDone! All images resized and added to {zip_filename}")
        files.download(zip_filename)

# Run the tool
resize_to_dimensions()

Please upload the images you want to resize:


Saving Muhammad Hafizh Nur Dzaki.jpg to Muhammad Hafizh Nur Dzaki (2).jpg
Enter maximum dimensions as Width x Height (e.g., 400x600): 400x600
Resizing Muhammad Hafizh Nur Dzaki (2).jpg to 400x600 (max bounds: 400x600)...
Done! Download starting...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>